# NILM-Engine — Colab GCS 학습 (EXP1~EXP4)

| 항목 | 경로 |
|------|------|
| 원천데이터 | `nilm/training_dev10/household_id=.../channel=.../date=.../` |
| 라벨데이터 | `nilm/labels/training.parquet` (5.9MB) |
| Train | house_011, 015, 016, 017, 033, 039, 054, 063 |
| Val | house_049 |
| Test | house_067 |

**실행 순서**: 1→2→3→4→5(선택) → **6~9 (학습)**

## 1. 환경 설치 & GCS 인증

In [ ]:
!pip install -q gudhi

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("GCS 인증 완료")

## 2. GCS 연결 검증

In [ ]:
!gcloud storage ls gs://ax-nilm-data-dhwang0803-us/nilm/training_dev10/ | head -5

In [ ]:
import gcsfs
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from pyarrow.fs import PyFileSystem, FSSpecHandler

gcs = gcsfs.GCSFileSystem()
_gcs_pa = PyFileSystem(FSSpecHandler(gcs))  # pyarrow 직접 읽기용

print("=== 원천데이터 스키마 ===")
raw_ds = ds.dataset(
    "ax-nilm-data-dhwang0803-us/nilm/training_dev10",
    filesystem=_gcs_pa, partitioning=["household_id", "channel", "date"],
)
print(raw_ds.schema)

print("\n=== 라벨 스키마 ===")
label_tbl = pq.read_table(
    "ax-nilm-data-dhwang0803-us/nilm/labels/training.parquet", filesystem=_gcs_pa
)
print(label_tbl.schema)
print(f"\n행수: {label_tbl.num_rows:,}")

In [ ]:
# 1일치 행수 — 2,592,000 (30Hz × 86400s) 이면 OK
t = pq.read_table(
    "ax-nilm-data-dhwang0803-us/nilm/training_dev10/household_id=house_011/channel=ch01/date=20231004/part-0.parquet",
    filesystem=_gcs_pa,
)
print(f"행수: {t.num_rows:,}")

## 3. 분할 & 경로 상수

In [ ]:
SPLIT = {
    "train": ["house_011", "house_015", "house_016", "house_017",
              "house_033", "house_039", "house_054", "house_063"],
    "val":   ["house_049"],
    "test":  ["house_067"],
}
BUCKET_PREFIX = "ax-nilm-data-dhwang0803-us/nilm/training_dev10"
LABEL_PATH    = "ax-nilm-data-dhwang0803-us/nilm/labels/training.parquet"
MODELS        = ["seq2point"] # 여기서 보인 모델만 선택해서 진행  ["seq2point", "bert4nilm", "cnn_tda"]

print(f"Train {len(SPLIT['train'])}개 / Val {len(SPLIT['val'])}개 / Test {len(SPLIT['test'])}개")

## 4. 리포지토리 설정

In [ ]:
import os, sys

REPO_DIR = "/content/ax_nilm"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dhwang0803-glitch/ax_nilm {REPO_DIR} -q
    print("클론 완료")
else:
    !git -C {REPO_DIR} pull -q && echo "최신화 완료"

SRC_DIR     = f"{REPO_DIR}/nilm-engine/src"
SCRIPTS_DIR = f"{REPO_DIR}/nilm-engine/scripts"
CFG_DIR     = f"{REPO_DIR}/nilm-engine/config"

for d in [SRC_DIR, SCRIPTS_DIR]:
    if d not in sys.path:
        sys.path.insert(0, d)
print("경로 설정 완료")

In [ ]:
import yaml

from acquisition.gcs_loader import (
    list_channels_gcs, get_house_start_date_gcs,
    load_channel_data_gcs, get_appliance_name_gcs,
    GCSNILMDataset,
)
from acquisition.preprocessor import PowerScaler
from train_model import (
    build_model, masked_weighted_mse, evaluate, train_one_epoch,
    compute_pos_weight, _NILMDatasetWithTDA, APPLIANCE_LABELS,
)

with open(f"{CFG_DIR}/train.yaml")   as f: TRAIN_CFG   = yaml.safe_load(f)
with open(f"{CFG_DIR}/dataset.yaml") as f: DATASET_CFG = yaml.safe_load(f)

print("모든 모듈 import 완료")

## 5. 데이터 탐색 (선택)

In [ ]:
print(f"{'house':12s} {'채널수':>6s}  {'시작일':>12s}")
print("-" * 36)
for house_id in SPLIT["train"] + SPLIT["val"] + SPLIT["test"]:
    channels = list_channels_gcs(gcs, house_id, BUCKET_PREFIX)
    try:
        start = get_house_start_date_gcs(gcs, house_id, bucket_prefix=BUCKET_PREFIX)
    except FileNotFoundError:
        start = "N/A"
    print(f"{house_id:12s} {len(channels):>6d}  {str(start):>12s}")

In [ ]:
# house_011 채널→가전 매핑
for ch in list_channels_gcs(gcs, "house_011", BUCKET_PREFIX):
    print(f"  {ch}: {get_appliance_name_gcs(gcs, 'house_011', ch, LABEL_PATH)}")

## 6. Google Drive 마운트 (체크포인트 영구 저장)

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

CKPT_DIR    = Path("/content/drive/MyDrive/nilm/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/nilm/results")
CACHE_DIR   = Path("/content/drive/MyDrive/nilm/cache")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"체크포인트 → {CKPT_DIR}")
print(f"결과       → {RESULTS_DIR}")
print(f"캐시       → {CACHE_DIR}")

In [ ]:
import gc
ws = DATASET_CFG["window"]["size"]
st = DATASET_CFG["window"]["stride"]
ec = DATASET_CFG["window"].get("event_context")
ss = DATASET_CFG["window"].get("steady_stride")

for exp_name, exp_cfg in TRAIN_CFG["experiments"].items():
    week = exp_cfg.get("week")
    print(f"\n[{exp_name}] week={week} 캐시 빌드 중...")

    # train: 해당 주차 / val: 전체 기간 고정 (run_exp_gcs와 동일 조건)
    for split_name, houses, split_week in [
        ("train", SPLIT["train"], week),
        ("val",   SPLIT["val"],   None),
    ]:
        base = GCSNILMDataset(
            houses, gcs_fs=gcs,
            bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
            window_size=ws, stride=st, week=split_week,
            event_context=ec, steady_stride=ss,
            fit_scaler=(split_name == "train"),
            cache_dir=CACHE_DIR,
            denoise=False,
        )
        print(f"  {split_name}(week={split_week}): {len(base):,} windows")

        if "cnn_tda" in MODELS:
            tda = _NILMDatasetWithTDA(base, cache_dir=CACHE_DIR)
            print(f"  {split_name} TDA: 완료")
            del tda

        del base
        gc.collect()

print("\n✅ 전체 캐시 완료 — 이후 run_exp_gcs는 Drive에서 로드합니다")


In [ ]:
import json, time
import torch
from torch.utils.data import DataLoader


def run_exp_gcs(exp_name: str, model_name: str,
                denoise: bool = True, tag: str = "") -> dict:
    """GCS 데이터로 단일 EXP/모델 학습. 결과를 Drive에 저장하고 metrics dict 반환.

    denoise : wavelet denoising 적용 여부. ablation 시 False로 호출.
    tag     : 비어 있으면 기존 파일명 그대로. ablation 시 "denoise_on" 등 suffix 지정.
              예) tag="denoise_on"  →  EXP1_cnn_tda_denoise_on.pt
    """
    exp_cfg = TRAIN_CFG["experiments"][exp_name]
    week = exp_cfg.get("week")
    ws   = DATASET_CFG["window"]["size"]
    st   = DATASET_CFG["window"]["stride"]
    ec   = DATASET_CFG["window"].get("event_context")
    ss   = DATASET_CFG["window"].get("steady_stride")
    bs   = TRAIN_CFG["training"]["batch_size"]
    ep   = TRAIN_CFG["training"]["epochs"]
    pat  = TRAIN_CFG["training"]["early_stopping_patience"]
    lr   = TRAIN_CFG["training"]["learning_rate"]
    _suffix = f"_{tag}" if tag else ""
    _label  = f"{exp_name}/{model_name}" + (f"[{tag}]" if tag else "")

    if exp_cfg.get("resume_from"):
        _prev_metrics = RESULTS_DIR / f'{exp_cfg["resume_from"]}_{model_name}{_suffix}_metrics.json'
        if _prev_metrics.exists():
            lr = json.load(open(_prev_metrics)).get("final_lr", lr)
            print(f"  └─ LR 이어받기: {lr:.2e}")
    wd   = TRAIN_CFG["optimizer"]["weight_decay"]
    lambda_mse = TRAIN_CFG["training"]["loss_weights"]["mse"]


    print(f"\n{'='*58}")
    print(f"  {_label}  week={week}  denoise={denoise}")
    print(f"{'='*58}")

    # ── 1. 데이터셋 ──────────────────────────────────────────────────────────
    print("  [1/4] 데이터셋 로딩...")

    resume_exp = exp_cfg.get("resume_from")
    prev_scaler = None
    if resume_exp:
        scaler_path = CKPT_DIR / f"{resume_exp}_{model_name}{_suffix}_scaler.json"
        if scaler_path.exists():
            prev_scaler = PowerScaler.load(scaler_path)
            print(f"  └─ scaler 로드: mean={prev_scaler.mean:.2f}W  std={prev_scaler.std:.2f}W")

    _ds_common = dict(
        gcs_fs=gcs,
        bucket_prefix=BUCKET_PREFIX,
        label_path=LABEL_PATH,
        window_size=ws, stride=st,
        event_context=ec, steady_stride=ss,
        cache_dir=CACHE_DIR,
        denoise=denoise,
    )
    # train: 해당 주차만 / val: 전체 기간 고정 (EXP 간 포화점 비교를 위해)
    _ds_train_kw = dict(**_ds_common, week=week)
    _ds_val_kw   = dict(**_ds_common, week=None)

    if prev_scaler is not None:
        base_train = GCSNILMDataset(SPLIT["train"], scaler=prev_scaler, **_ds_train_kw)
        base_val   = GCSNILMDataset(SPLIT["val"],   scaler=prev_scaler, **_ds_val_kw)
    else:
        base_train = GCSNILMDataset(SPLIT["train"], fit_scaler=True,          **_ds_train_kw)
        base_val   = GCSNILMDataset(SPLIT["val"],   scaler=base_train.scaler, **_ds_val_kw)

    if model_name == "cnn_tda":
        train_ds = _NILMDatasetWithTDA(base_train, cache_dir=CACHE_DIR)
        val_ds   = _NILMDatasetWithTDA(base_val,   cache_dir=CACHE_DIR)
    else:
        train_ds, val_ds = base_train, base_val

    train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=2, pin_memory=False)
    val_dl   = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, pin_memory=False)
    print(f"  train={len(train_ds):,}  val={len(val_ds):,} windows")

    # ── 2. 모델 ──────────────────────────────────────────────────────────────
    print("  [2/4] 모델 초기화...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  device: {device}")
    model = build_model(model_name, ws).to(device)

    if resume_exp:
        prev_ckpt = CKPT_DIR / f"{resume_exp}_{model_name}{_suffix}.pt"
        if prev_ckpt.exists():
            _ckpt = torch.load(prev_ckpt, map_location=device, weights_only=True)
            _state = _ckpt["model_state"] if isinstance(_ckpt, dict) and "model_state" in _ckpt else _ckpt
            model.load_state_dict(_state)
            print(f"  └─ 모델 로드: {prev_ckpt.name}")
        else:
            print(f"  └─ 경고: {prev_ckpt.name} 없음 — 처음부터 학습")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, "min",
        factor=TRAIN_CFG["scheduler"]["factor"],
        patience=TRAIN_CFG["scheduler"]["patience"],
    )
    amp_scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    # per-appliance loss scale (구조 C)
    _app_index = {name: i for i, name in enumerate(APPLIANCE_LABELS)}
    _scale_cfg = TRAIN_CFG.get("appliance_loss_scale", {})
    appliance_scale = torch.ones(len(APPLIANCE_LABELS), device=torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    for _name, _s in _scale_cfg.items():
        if _name in _app_index:
            appliance_scale[_app_index[_name]] = float(_s)
            print(f"  appliance_loss_scale [{_name}]: ×{_s}")

    pos_weight_max = float(TRAIN_CFG["training"].get("pos_weight_max", 20.0))
    pos_weight = None
    if model_name == "cnn_tda":
        print("  pos_weight 계산 중...")
        pos_weight = compute_pos_weight(train_dl, device, max_weight=pos_weight_max)
        for name, pw in zip(APPLIANCE_LABELS, pos_weight.cpu().tolist()):
            print(f"    {name}: {pw:.2f}")

    # ── 3. 학습 루프 ─────────────────────────────────────────────────────────
    print("  [3/4] 학습 시작...")
    best_score, best_mae, best_cls_thresholds, best_vm, best_state, no_improve = \
        (-float("inf"), float("inf")), float("inf"), None, None, None, 0
    t0 = time.perf_counter()

    for epoch in range(1, ep + 1):
        loss = train_one_epoch(model, train_dl, optimizer, model_name, device,
                               amp_scaler=amp_scaler, pos_weight=pos_weight,
                               lambda_mse=lambda_mse, appliance_scale=appliance_scale)
        vm   = evaluate(model, val_dl, model_name, device)
        scheduler.step(vm["mae"])
        lr_now = optimizer.param_groups[0]["lr"]
        f1_cls_str = f"  f1_cls={vm['f1_cls']:.3f}" if vm.get("f1_cls") is not None else ""
        print(f"  ep {epoch:3d}/{ep}  loss={loss:.4f}  "
              f"val_mae={vm['mae']:.2f}W  f1={vm['f1']:.3f}{f1_cls_str}  lr={lr_now:.2e}")

        _f1_cls = vm.get("f1_cls") or 0.0
        _score  = (_f1_cls, -vm["mae"])
        if _score > best_score or best_state is None:
            best_score          = _score
            best_mae            = vm["mae"]
            best_cls_thresholds = vm.get("best_cls_thresholds")
            best_vm             = vm
            best_state          = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve          = 0
        else:
            no_improve += 1
            if no_improve >= pat:
                print(f"  조기 종료 ({pat} epoch 개선 없음)")
                break

    elapsed = time.perf_counter() - t0

    # ── 4. 저장 ──────────────────────────────────────────────────────────────
    print("  [4/4] 체크포인트 & 메트릭 저장...")
    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    torch.save({"model_state": model.state_dict(), "best_cls_thresholds": best_cls_thresholds},
               CKPT_DIR / f"{exp_name}_{model_name}{_suffix}.pt")
    if base_train.scaler is not None:
        base_train.scaler.save(CKPT_DIR / f"{exp_name}_{model_name}{_suffix}_scaler.json")

    fm = best_vm if best_vm is not None else vm
    fm.update({"exp": exp_name, "model": model_name, "week": week,
               "tag": tag, "denoise": denoise,
               "val_weeks": "all", "training_time_s": round(elapsed, 1),
               "final_lr": optimizer.param_groups[0]["lr"]})
    with open(RESULTS_DIR / f"{exp_name}_{model_name}{_suffix}_metrics.json", "w") as f:
        json.dump(fm, f, ensure_ascii=False, indent=2)

    f1_cls_str = f"  F1_cls={fm['f1_cls']:.3f}" if fm.get("f1_cls") is not None else ""
    print(f"  ✅  val MAE={fm['mae']:.2f}W  RMSE={fm['rmse']:.2f}W  "
          f"SAE={fm['sae']:.4f}  F1={fm['f1']:.3f}{f1_cls_str}  ({elapsed:.0f}s)")

    # per-appliance RMSE 요약 (문제 5)
    _pa = fm.get("per_appliance", {})
    if _pa:
        print("  [per-appliance RMSE]")
        _rows = [(n, m["rmse"], m["mae"]) for n, m in _pa.items()
                 if m.get("rmse") is not None and m.get("mae") is not None]
        _rows.sort(key=lambda x: -(x[1] / max(x[2], 1e-8)))
        for _n, _r, _m in _rows:
            _ratio = _r / max(_m, 1e-8)
            _flag = "  ⚠️" if _ratio > 2.0 else ""
            print(f"    {_n}: RMSE={_r:.1f}W  MAE={_m:.1f}W  ({_ratio:.1f}x){_flag}")
    return fm


def write_results_md(exp_name: str) -> None:
    """EXP 결과 MD를 RESULTS_DIR에 생성 (Drive에 저장됨)."""
    from datetime import datetime

    exp_cfg  = TRAIN_CFG["experiments"][exp_name]
    week     = exp_cfg.get("week", "?")
    prev_exp = exp_cfg.get("resume_from")

    exp_ms  = {}
    prev_ms = {}
    for m in MODELS:
        p = RESULTS_DIR / f"{exp_name}_{m}_metrics.json"
        exp_ms[m] = json.load(open(p)) if p.exists() else None
        pp = RESULTS_DIR / f"{prev_exp}_{m}_metrics.json"
        prev_ms[m] = (json.load(open(pp)) if pp.exists() else None) if prev_exp else None

    week_start = (week - 1) * 7 + 1 if isinstance(week, int) else "?"
    week_end   = week * 7            if isinstance(week, int) else "?"
    lines = [
        f"# {exp_name} 실험 결과", "",
        "| 항목 | 값 |", "|------|-----|",
        f"| 학습 주차 | week {week} (house별 시작일 기준 {week_start}~{week_end}일차) |",
        f"| 이전 체크포인트 | {prev_exp or '처음부터'} |",
        f"| Val 기간 | house_049 전체 기간 고정 |",
        f"| 기록 일시 | {datetime.now().strftime('%Y-%m-%d %H:%M')} |",
        "", "---", "",
        "## Val 성능 비교", "",
        "| 모델 | MAE (W) | RMSE (W) | SAE | F1 | F1_cls |",
        "|------|---------|----------|-----|----|--------|",
    ]
    for m in MODELS:
        md = exp_ms.get(m)
        if md:
            f1_cls = f"{md['f1_cls']:.3f}" if md.get("f1_cls") is not None else "—"
            lines.append(f"| {m} | {md['mae']:.2f} | {md['rmse']:.2f} | {md['sae']:.4f} | {md['f1']:.3f} | {f1_cls} |")
        else:
            lines.append(f"| {m} | — | — | — | — | — |")

    lines += ["", "---", "", "## 이전 EXP 대비 개선율 (Val MAE)"]
    if not prev_exp:
        lines += ["", "> 첫 번째 실험 — 비교 대상 없음"]
    else:
        lines += ["", "| 모델 | 이전 MAE | 현재 MAE | 개선율 |",
                  "|------|---------|---------|--------|"]
        for m in MODELS:
            cur, prv = exp_ms.get(m), prev_ms.get(m)
            if cur and prv:
                imp = (prv["mae"] - cur["mae"]) / (prv["mae"] + 1e-8) * 100
                trend = "↓" if imp > 0 else "↑"
                lines.append(f"| {m} | {prv['mae']:.2f}W | {cur['mae']:.2f}W | {trend} {abs(imp):.1f}% |")
            else:
                lines.append(f"| {m} | — | — | — | — |")

    lines += ["", "---", "", "## 메모", "",
              "> 특이사항, 하이퍼파라미터 변경 내역 등을 여기에 기록하세요.", ""]

    md_path = RESULTS_DIR / f"{exp_name}_results.md"
    md_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"MD 생성: {md_path}")


print("함수 정의 완료 — run_exp_gcs / write_results_md")

## 8. EXP1 실행

> ⚠️ **메모리**: 8 houses × 7일 × 30Hz ≈ 1.4억 샘플.  
> **시간**: GPU 기준 seq2point ~20min, bert4nilm ~30min, cnn_tda ~60min (epoch당).

In [ ]:
results = {}

for model_name in MODELS:
    results[("EXP1", model_name)] = run_exp_gcs("EXP1", model_name, denoise=False)

write_results_md("EXP1")


## 9. EXP2~4 연속 실행 (포화점 자동 감지)

이전 EXP 대비 Val MAE 평균 개선율 < `saturation_threshold` (5%) 이면 자동 중단.

In [ ]:
import json as _json

# results 미정의 시 Drive 저장 메트릭으로 복원 (런타임 재시작 후 재실행 안전)
if "results" not in globals():
    results = {}
    for _exp in ["EXP1", "EXP2", "EXP3", "EXP4"]:
        for _m in MODELS:
            _p = RESULTS_DIR / f"{_exp}_{_m}_metrics.json"
            if _p.exists():
                results[(_exp, _m)] = _json.load(open(_p))
                print(f"  복원: ({_exp}, {_m})")

for exp_name in ["EXP2", "EXP3", "EXP4"]:
    for model_name in MODELS:
        if (exp_name, model_name) not in results:
            results[(exp_name, model_name)] = run_exp_gcs(exp_name, model_name, denoise=False)
        else:
            print(f"  스킵 (Drive 메트릭 로드): {exp_name}/{model_name}")
    write_results_md(exp_name)


## 10. 결과 요약 조회

In [ ]:
print(f"{'EXP':6s} {'모델':14s} {'MAE':>8s} {'RMSE':>8s} {'SAE':>8s} {'F1':>6s}")
print("-" * 56)
for (exp_name, model_name), m in sorted(results.items()):
    if m:
        print(f"{exp_name:6s} {model_name:14s} {m['mae']:8.2f} {m['rmse']:8.2f} {m['sae']:8.4f} {m['f1']:6.3f}")

print(f"\nMD 파일 위치: {RESULTS_DIR}/EXP*_results.md")

## 10-1. Per-Appliance RMSE/MAE 분석 (문제 #5 진단)

RMSE/MAE 비율이 3x 이상인 가전 = 간헐적 대형 오차 발생 가전.
학습 EXP가 진행될수록 비율이 낮아지면 데이터 부족이 원인.

In [ ]:
# ── per-appliance RMSE/MAE 분석 ──────────────────────────────────────────
# results dict 또는 Drive 저장 JSON에서 로드
import json as _json
from pathlib import Path

def show_appliance_rmse(exp_name, model_name, results_dir=None):
    """RMSE/MAE 비율 순으로 가전 목록 출력"""
    if results_dir is None:
        results_dir = RESULTS_DIR
    # results dict 우선, 없으면 Drive JSON 로드
    m = results.get((exp_name, model_name)) if 'results' in dir() else None
    if m is None:
        p = Path(results_dir) / f'{exp_name}_{model_name}_metrics.json'
        if not p.exists():
            print(f'  파일 없음: {p.name}')
            return
        m = _json.load(open(p))

    pa = m.get('per_appliance', {})
    rows = [
        (name, v['rmse'], v['mae'])
        for name, v in pa.items()
        if v.get('rmse') is not None and v.get('mae') is not None and v['mae'] > 0
    ]
    rows.sort(key=lambda x: x[1] / x[2], reverse=True)  # 비율 내림차순

    print(f'\n{'='*58}')
    print(f'  {exp_name} / {model_name} — per-appliance RMSE 분석')
    print(f'{'='*58}')
    print(f"  {'가전':<22} {'RMSE':>8} {'MAE':>8} {'비율':>6}")
    print('  ' + '-' * 48)
    for name, r, a in rows:
        flag = ' ⚠️' if r / a >= 2.5 else ''
        print(f"  {name:<22} {r:>8.1f} {a:>8.1f} {r/a:>6.2f}x{flag}")

# 분석할 EXP/모델 지정
for exp in ['EXP1', 'EXP2', 'EXP3', 'EXP4']:
    show_appliance_rmse(exp, 'cnn_tda')


## A-0. Denoise Ablation — wavelet denoising ON vs OFF

**목적**: denoising이 F1에 기여하는지, transient 정보를 뭉개는지 판별  
**변수**: `denoise=True` vs `denoise=False` (나머지 모든 조건 동일)  
**대상**: `cnn_tda` / `EXP1` (다른 변경 적용 전 baseline에서 단독 측정)  
**판단 기준**:
- F1(전체) 차이 < 0.01 → 효과 없음, OFF 유지 고려
- F1(전체) ON > OFF → 유지
- 헤어드라이기·전기포트 등 fast transient 가전의 per-appliance F1이 OFF에서 상승 → transient 손상 의심

## 11. 최종 Test 평가 (전 EXP 완료 후 1회)

> 모든 EXP가 끝난 뒤 **단 한 번** 실행. 중간에 test 결과를 보고 의사결정하면 leakage.
> 마지막으로 저장된 체크포인트(가장 많이 학습된 EXP)를 로드해 test set으로 평가한다.

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader

ws = DATASET_CFG["window"]["size"]
st = DATASET_CFG["window"]["stride"]
ec = DATASET_CFG["window"].get("event_context")
ss = DATASET_CFG["window"].get("steady_stride")
bs = TRAIN_CFG["training"]["batch_size"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 마지막으로 완료된 EXP 탐색
completed = sorted([k for k, v in results.items() if v], reverse=True)
last_exp = completed[0][0] if completed else list(TRAIN_CFG["experiments"].keys())[-1]

print(f"최종 Test 평가 — {last_exp} 체크포인트 사용")
print("=" * 58)

_ds_kwargs = dict(
    gcs_fs=gcs,
    bucket_prefix=BUCKET_PREFIX,
    label_path=LABEL_PATH,
    window_size=ws, stride=st, week=None,  # test도 전체 기간
    event_context=ec, steady_stride=ss,
    cache_dir=CACHE_DIR,
    denoise=False,
)

for model_name in MODELS:
    ckpt_path = CKPT_DIR / f"{last_exp}_{model_name}.pt"
    scaler_path = CKPT_DIR / f"{last_exp}_{model_name}_scaler.json"
    if not ckpt_path.exists():
        print(f"  [{model_name}] 체크포인트 없음 — 스킵")
        continue

    scaler = PowerScaler.load(scaler_path) if scaler_path.exists() else None
    base_test = GCSNILMDataset(SPLIT["test"], scaler=scaler, **_ds_kwargs)
    test_ds = _NILMDatasetWithTDA(base_test, cache_dir=CACHE_DIR) if model_name == "cnn_tda" else base_test
    test_dl = DataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=2, pin_memory=False)

    model = build_model(model_name, ws).to(device)
    _ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    _state = _ckpt["model_state"] if isinstance(_ckpt, dict) and "model_state" in _ckpt else _ckpt
    _cls_thr = np.array(_ckpt["best_cls_thresholds"]) if isinstance(_ckpt, dict) and _ckpt.get("best_cls_thresholds") is not None else None
    model.load_state_dict(_state)

    # val에서 구한 cls_thresholds를 고정 — test set에서 임계값 탐색 금지 (leakage 방지)
    tm = evaluate(model, test_dl, model_name, device, cls_thresholds=_cls_thr)
    tm.update({"exp": last_exp, "model": model_name, "week": "all", "split": "test"})

    out_path = RESULTS_DIR / f"{last_exp}_{model_name}_test_metrics.json"
    with open(out_path, "w") as f:
        json.dump(tm, f, ensure_ascii=False, indent=2)

    print(f"  {model_name:14s}  MAE={tm['mae']:.2f}W  RMSE={tm['rmse']:.2f}W  "
          f"SAE={tm['sae']:.4f}  F1={tm['f1']:.3f}  → {out_path.name}")
